# Session 4: One interface to rule them all

**Week 2 · Session 2 · Project version v1**

Session 3 ended with the CTO asking for a Plan B vendor by Friday. v0.2 is written in Google's SDK
from top to bottom, so switching would mean rewriting it. Tonight we move it onto **LangChain**, a
framework that puts one interface in front of many vendors, and switch on **LangSmith**, so every
call the assistant makes leaves a record you can open.

> *A framework is a trade: you hand over direct control and get back a standard interface. Your job
> is to know exactly where the interface stops.*

<img src="figures/session-map.png" width="900"
     alt="Session 4 on one page. A top strip corrects session 3: Gemini's response_schema field rejects
     additionalProperties while response_json_schema accepts it. Part A, the interface: three routes
     to vendor neutrality, one provider string, five providers in one loop, and the seam where
     validation context is lost and kept. Part B, production: switch versus fallback, batching under a
     rate limiter, cache reads through the interface. Part C, the evidence: LangSmith switched on with
     two environment variables, one ticket's run with its steps, tokens and cost.">

| Section | What it covers |
|---|---|
| 1 | The trade, and three ways to make it |
| 2 | One string, one call: the standard interface |
| 3 | Five providers, one loop: where the interface stops |
| 4 | Structured output, and the check the framework can't do |
| 5 | Plan B: switch or fall back |
| 6 | Many calls: batching, rate limits, caching |
| 7 | LangSmith: two environment variables, zero code changes |
| 8 | The ledger: what the port bought |
| 9 | Project v1 |
| 10 | Wrap-up, and Ayesha's order |

## Setup

The imports move from `google.genai` to `langchain`. Keys: Gemini as before; **Groq** and
**LangSmith** were the pre-work (two minutes each, sections 3.5 and 7.1 if you skipped it). OpenAI and
Anthropic are optional: their outputs are saved in this notebook. **Ollama** is optional and free, and
runs on your laptop (section 3.6).

The table this cell prints decides what runs on your machine. Later cells skip a provider you can't
run, with one line, instead of failing.

In [1]:
import hashlib
import os
import subprocess
import sys
import time
import urllib.request
import warnings
from pathlib import Path

import yaml
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, ConfigDict, ValidationError

sys.path.insert(0, str(Path.cwd() / "project"))      # the v1 snapshot beside this notebook
from config import settings                            # noqa: E402
from schemas import Ticket, TicketFacts, TriageResult  # noqa: E402
from utils import ask_json, load_inbox                 # noqa: E402
import triage as v1                                    # noqa: E402

TICKETS = load_inbox()
by_id = {t.id: t for t in TICKETS}

# The five provider strings this notebook uses, and the key each one needs.
PROVIDERS = {
    "google_genai:gemini-3.5-flash-lite": "GEMINI_API_KEY",
    "groq:openai/gpt-oss-20b": "GROQ_API_KEY",
    "openai:gpt-5.4-nano": "OPENAI_API_KEY",
    "anthropic:claude-haiku-4-5-20251001": "ANTHROPIC_API_KEY",
    "ollama:llama3.2:3b": None,                # local: no key, but Ollama must be running
}


def ollama_running() -> bool:
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=1)
        return True
    except OSError:
        return False


AVAILABLE = {}
for model_id, key_name in PROVIDERS.items():
    AVAILABLE[model_id] = ollama_running() if key_name is None else bool(os.getenv(key_name))
    print(f"{'yes' if AVAILABLE[model_id] else ' - '}   {model_id}")

print(f"\nLangSmith tracing: {'on, project ' + settings.LANGSMITH_PROJECT if settings.LANGSMITH_TRACING else 'off'}")

yes   google_genai:gemini-3.5-flash-lite
yes   groq:openai/gpt-oss-20b
yes   openai:gpt-5.4-nano
yes   anthropic:claude-haiku-4-5-20251001
yes   ollama:llama3.2:3b

LangSmith tracing: on, project shopwise-cse


## The promise from last session, corrected

Session 3's appendix D said no single schema satisfies Gemini and OpenAI: Gemini rejects
`additionalProperties: false`, and OpenAI's strict mode requires it. Check that claim properly. Same
schema, and the two different fields Gemini's SDK offers for it:

In [2]:
from google import genai              # the raw SDK, one last time


class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")      # puts "additionalProperties": false in the schema
    category: str


client = genai.Client()
ticket_text = by_id["TCK-009"].as_block()

try:
    client.models.generate_content(model=settings.MODEL, contents=ticket_text, config={
        "response_mime_type": "application/json", "response_schema": Strict})
except Exception as err:
    print("response_schema      ->", type(err).__name__, str(err)[:80])

r = client.models.generate_content(model=settings.MODEL, contents=ticket_text, config={
    "response_mime_type": "application/json", "response_json_schema": Strict.model_json_schema()})
print("response_json_schema ->", r.text)

response_schema      -> ClientError 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Invalid JSON payload r


response_json_schema -> {
  "category": "Billing"
}


The keyword was never the problem. One of Gemini's two schema fields rejects it and the other accepts
it. Last session I tested one field and wrote down a rule. That's exactly what session 2 warned
about: advice, mine included, is a hypothesis until you measure it. Session 3's notebook is corrected.

The CTO's question still stands, though. How much of v0.2 is written for Google's SDK?

In [3]:
GOOGLE_SHAPED = ["genai", "types.", "errors.", "generate_content", "HttpOptions", "response_schema",
                 "finish_reason", "candidates", "system_instruction", "stop_sequences",
                 "max_output_tokens="]

count = 0
for path in [Path("../session-03/project/utils.py"), Path("../session-03/project/triage.py"),
             Path("../session-03/score.py")]:
    for number, line in enumerate(path.read_text().splitlines(), start=1):
        code_part = line.split("#")[0]
        if any(word in code_part for word in GOOGLE_SHAPED):
            count += 1
            print(f"{path.name:9} {number:4}  {line.strip()[:78]}")

print(f"\n{count} lines written for one vendor's SDK")

utils.py    10  from google import genai
utils.py    11  from google.genai import types
utils.py    22  def build_client() -> genai.Client:
utils.py    30  return genai.Client(
utils.py    31  http_options=types.HttpOptions(
utils.py    33  retry_options=types.HttpRetryOptions(
utils.py    50  config = types.GenerateContentConfig(
utils.py    51  system_instruction=system,
utils.py    53  response_schema=schema,
utils.py    54  max_output_tokens=settings.MAX_OUTPUT_TOKENS,
utils.py    58  response = client.models.generate_content(
utils.py    61  reason = response.candidates[0].finish_reason if response.candidates else "unk
utils.py    62  raise ModelReturnedNothing(f"no text (finish_reason={reason})")
triage.py   27  from google.genai import errors, types
triage.py   87  response = client.models.generate_content(
triage.py   90  config=types.GenerateContentConfig(
triage.py   91  system_instruction=reply_system(date.today()) + steer,
triage.py   92  stop_sequences=["</reply>"],
triage

Those lines are spread through every function that does real work: the client, both kinds of call,
the error handling, the token counts. There's no seam to cut along, because in v0.2 the vendor isn't a
component, it's in every line. You could write your own adapter layer, and then maintain a framework
with exactly one user. Section 1 looks at the three ways industry does it instead.

One more thing before we do. Through LangChain, that strict schema simply works on every vendor
(section 4), and nothing tells you which of Gemini's two fields it chose. The framework knew
something you didn't, and didn't mention it. That's the trade in one example.

## 1. The trade, and three ways to make it

"Change vendor without rewriting the program" has three answers in industry today.

<img src="figures/three-routes.png" width="900"
     alt="Three routes to vendor neutrality side by side. One: an OpenAI-compatible endpoint, the
     OpenAI SDK with another base_url, cheap but each vendor's own features leak through extra_body
     and errors and structured output still differ. Two: a gateway such as LiteLLM or OpenRouter in the
     middle, giving routing, fallbacks and spend limits at the network layer, at the price of another
     hop and another company handling every message. Three: a framework, LangChain, with integration
     packages behind init_chat_model, giving one message type, per-vendor structured output, tracing
     and the agent stack, at the price of one more layer between you and the error. A band underneath
     states the criterion: use the framework when the shared interface saves more than the abstraction
     costs.">

**Docs:** [LangChain philosophy](https://docs.langchain.com/oss/python/langchain/philosophy) ·
[Gemini's OpenAI compatibility](https://ai.google.dev/gemini-api/docs/openai) ·
[Groq's](https://console.groq.com/docs/openai) ·
gateways: [LiteLLM](https://docs.litellm.ai/docs/routing),
[OpenRouter](https://openrouter.ai/docs/guides/routing/model-fallbacks)

Route 1 costs nothing to try. Gemini and Groq both answer the OpenAI SDK if you point it at their
URL:

In [4]:
from openai import OpenAI

ENDPOINTS = [   # (base_url, key name, model)
    ("https://generativelanguage.googleapis.com/v1beta/openai/", "GEMINI_API_KEY", settings.MODEL),
    ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "openai/gpt-oss-20b"),
]

for base_url, key_name, model_name in ENDPOINTS:
    if not os.getenv(key_name):
        print(f"{model_name}: skipped, no {key_name}")
        continue
    sdk = OpenAI(base_url=base_url, api_key=os.getenv(key_name))
    r = sdk.chat.completions.create(model=model_name, messages=[
        {"role": "user", "content": "In five words: what does a support ticket need?"}])
    print(f"{model_name:24} {r.choices[0].message.content.strip()}")

gemini-3.5-flash-lite    Clear issue and contact info.


openai/gpt-oss-20b       Subject, description, priority, contact, attachments.


Same code, two vendors, no framework. For one vendor and one kind of call that's a perfectly good
answer, and it's why sessions 2 and 3 used the raw SDK. Where it stops: Gemini's own settings
(thinking, caching, safety) have to be smuggled through `extra_body`; Groq rejects some OpenAI
parameters with a 400; Ollama's version has no `tool_choice`; and errors, structured output and
tracing are still yours to handle per vendor.

The docs' criterion is the one to remember: **use the framework when the shared interface saves more
than the abstraction costs.** For us the deciding facts are five providers tonight, and tools, agents,
graphs and memory from session 5 onwards.

### 1.1 What you installed

LangChain is several packages. You import from `langchain`; the vendor's own SDK is still there, one
level down.

In [5]:
from importlib.metadata import version

for package in ["langchain", "langchain-core", "langgraph", "langsmith", "langchain-google-genai",
                "langchain-groq", "langchain-openai", "langchain-anthropic", "langchain-ollama"]:
    print(f"{package:24} {version(package)}")

model = init_chat_model(settings.model_id)
print("\nthe vendor's SDK, one level down:", type(model.client))

langchain                1.3.14
langchain-core           1.6.1
langgraph                1.2.10
langsmith                0.10.17
langchain-google-genai   4.3.2
langchain-groq           1.1.3
langchain-openai         1.6.0
langchain-anthropic      1.7.0
langchain-ollama         1.1.0

the vendor's SDK, one level down: <class 'google.genai.client.Client'>


| Package | Role |
|---|---|
| `langchain` | what you import: `init_chat_model`, messages, and from session 5 `create_agent` |
| `langchain-core` | the shared interfaces every integration implements |
| `langchain-google-genai`, `-groq`, `-openai`, `-anthropic`, `-ollama` | one integration per vendor, wrapping that vendor's SDK |
| `langgraph` | explicit control flow, session 10 |
| `langsmith` | tracing, section 7 |

A tutorial that uses `LLMChain`, `AgentExecutor`, `from langchain_community.chat_models import ...` or
`reply.text()` was written before LangChain v1 (October 2025). The full card is in section 10.

## 2. One string, one call

LangChain is a set of building blocks, each with one interface whatever the vendor underneath.
Tonight uses three of them: **models**, **messages** and **structured output**, plus LangSmith to
record what they do. The rest arrive one session at a time and plug into the same interface: tools
and agents next session, then memory, middleware and retrieval. From session 10, LangGraph wires
them into explicit flows, and from session 13 the assistant becomes a product: an API, a UI, a
deployment.

<img src="figures/langchain-map.png" width="900"
     alt="LangChain's building blocks and when the course uses each. A grid of component tiles: models,
     messages and structured output marked tonight; tools and agents in session 5; memory in session 6;
     middleware in session 7; loaders, splitters and embeddings in session 8; vector stores and
     retrievers in sessions 8 and 9. Under them a wide LangGraph band for control flow, graphs, state,
     interrupts and long-term memory, sessions 10 to 12. Down the right side a LangSmith column spanning
     everything: tracing tonight, evaluation and experiments in session 9, the prompt registry in
     session 16. Along the bottom the product path: an API in session 13, a UI in session 15, deployed
     in session 16.">

This section redoes everything sessions 2 and 3 did with Google's SDK, on the standard interface.

**Docs:** [component architecture](https://docs.langchain.com/oss/python/langchain/component-architecture) ·
[models](https://docs.langchain.com/oss/python/langchain/models) ·
[`init_chat_model` reference](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model) ·
[messages](https://docs.langchain.com/oss/python/langchain/messages)

### 2.1 Choosing a model: the provider string

A model is named by one string, `provider:model`. The provider IDs are listed on the
[providers page](https://docs.langchain.com/oss/python/integrations/providers/overview); each
provider's model names are on its [chat integration page](https://docs.langchain.com/oss/python/integrations/chat)
and in the vendor's own docs.

In [ ]:
model = init_chat_model("google_genai:gemini-3.5-flash-lite")
reply = model.invoke("In one sentence: what is a support ticket?")
print(reply.text)

A support ticket is a digital record created to track, manage, and resolve a customer's or user's problem, question, or request.


The string is split at the **first** colon, so `ollama:llama3.2:3b` means provider `ollama`, model
`llama3.2:3b`. The project never spells a model out at a call site: it passes `settings.model_id`,
built from `SHOPWISE_PROVIDER` and `SHOPWISE_MODEL`.

Two traps here cost people an evening. The first is a hyphen, which is how the pip package spells it:

In [7]:
try:
    init_chat_model("google-genai:gemini-3.5-flash-lite")
except ValueError as err:
    print(str(err)[:330], "...")

Unable to infer model provider for model='google-genai:gemini-3.5-flash-lite'. Please specify 'model_provider' directly.

Supported providers: anthropic, anthropic_bedrock, azure_ai, azure_openai, baseten, bedrock, bedrock_converse, cohere, deepseek, fireworks, google_anthropic_vertex, google_genai, google_vertexai, groq, huggin ...


It reads like a complaint about the model. It isn't: the list of supported providers is the useful
part, and none of them contains a hyphen. The second trap is worse, because it half-works: leave the
provider out.

In [8]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    try:
        init_chat_model("gemini-3.5-flash-lite")
    except ImportError as err:
        print("ImportError:", err)

for w in caught:
    print("warning:", str(w.message)[:150])

ImportError: Initializing ChatVertexAI requires the langchain-google-vertexai package. Please install it with `pip install langchain-google-vertexai`


Without a provider, LangChain guesses **Vertex AI**: a different Google product, with different
billing and different credentials. Here it fails because that integration isn't installed; on a
machine where it is, it asks for Google Cloud credentials instead, which sends you debugging the wrong
thing. v1's `config.py` refuses a bad provider at startup:

In [9]:
from config import Settings

try:
    Settings(PROVIDER="google-genai")
except ValidationError as err:
    print(err.errors()[0]["msg"])

Value error, 'google-genai': provider IDs use underscores and no colon, e.g. google_genai, groq, openai, anthropic, ollama


### 2.2 The reply

`invoke` returns an `AIMessage`. Three ways to read it:

In [10]:
print("text:          ", reply.text)
print("content:       ", type(reply.content).__name__, "of", len(reply.content), "block(s)")
print("content_blocks:", [block["type"] for block in reply.content_blocks])
print("usage:         ", reply.usage_metadata)

text:           A support ticket is a digital record created to track, manage, and resolve a customer's or user's problem, question, or request.
content:        list of 1 block(s)
content_blocks: ['text']
usage:          {'input_tokens': 11, 'output_tokens': 30, 'total_tokens': 41, 'input_token_details': {'cache_read': 0}}


| | What it is |
|---|---|
| `.text` | the text, joined. A property: the old `.text()` still works but warns |
| `.content` | what the vendor sent. On Gemini 3 that's a **list** of blocks, so `reply.content.strip()`, common in older tutorials, raises `AttributeError` |
| `.content_blocks` | the same thing in LangChain's standard shape: text, reasoning, images, tool calls |
| `.usage_metadata` | `input_tokens`, `output_tokens`, `total_tokens`: the same names on every vendor. Session 2's `candidates_token_count` was Google's name |

### 2.3 Messages

Session 2 built a conversation by hand as a list of turns and resent the whole list every time.
That list is exactly how LangChain represents a conversation.

In [ ]:
from langchain.messages import AIMessage, HumanMessage, SystemMessage

conversation = [
    SystemMessage("You are ShopWise's support assistant. Answer in one sentence."),
    HumanMessage("My order is ORD-5002."),
    AIMessage("Thanks, I've noted ORD-5002."),
    HumanMessage("What was my order number?"),
]
print(model.invoke(conversation).text)

# The same conversation as (role, text) pairs, the spelling the project uses.
as_pairs = [("system", "You are ShopWise's support assistant. Answer in one sentence."),
            ("human", "My order is ORD-5002."),
            ("ai", "Thanks, I've noted ORD-5002."),
            ("human", "What was my order number?")]
            
as_pairs_openai_format = [
    {"role": "system", "content": "You are ShopWise's support assistant. Answer in one sentence."},
    {"role": "user", "content": "My order is ORD-5002."},
    {"role": "assistant", "content": "Thanks, I've noted ORD-5002."},
    {"role": "user", "content": "What was my order number?"}
]
print(model.invoke(as_pairs).text)

Your order number is ORD-5002.


Your order number is ORD-5002.


The model is still stateless: it knows `ORD-5002` only because we sent it again. The system message is
session 2's system slot; the two-slot layout carries over unchanged. Import messages from
`langchain.messages`. The older `langchain_core.messages` path still works and returns the same
classes, which is why you'll keep seeing it in tutorials.

### 2.4 Parameters: the ones that travel, and the ones that don't

**Docs:** [parameters](https://docs.langchain.com/oss/python/langchain/models#parameters)

In [12]:
model = init_chat_model(settings.model_id, timeout=30, max_retries=3, max_tokens=200)
print(model.invoke("Reply with the word ready, then a full stop, then the word now.", stop=["."]).text)

Ready


| Parameter | Means | Was, in v0.2 |
|---|---|---|
| `timeout` | seconds to wait for one call | `HttpOptions(timeout=...)`, in **milliseconds** |
| `max_retries` | attempts, counting the first, for 408, 429, 5xx, timeouts and dropped connections, with backoff from 1 s up to 60 s | the `HttpRetryOptions` session 3 configured, which replaced session 2's hand-written loop |
| `max_tokens` | cap on the reply | `max_output_tokens` |
| `stop` | stop before this text | `stop_sequences` |

These mean the same on every provider. Vendor-specific settings are passed through to the vendor's
own class, under the vendor's own name:

In [13]:
thinker = init_chat_model(settings.model_id, thinking_level="high", include_thoughts=True)
r = thinker.invoke("3 kettles at 2,400 taka each, 10% off the total. What is paid? Number only.")
print([block["type"] for block in r.content_blocks], r.usage_metadata["output_token_details"], r.text)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    init_chat_model(settings.model_id, temperature=0.2).invoke("Say hi.")
print([str(w.message) for w in caught if "sampling" in str(w.message)])

['reasoning', 'text'] {'reasoning': 275} 6480


["Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored."]


`thinking_level` is Gemini's name (session 2, section 3.4). OpenAI calls the same idea
`reasoning_effort` and Anthropic `thinking`. The interface standardises the call, not the knobs. The
reasoning comes back as a `reasoning` content block, and its tokens are counted separately.

And `temperature` is ignored with a warning: session 2 said not to tune it on Gemini 3, and for this
model LangChain carries that rule in a list of its own.

### 2.5 Streaming

**Docs:** [stream](https://docs.langchain.com/oss/python/langchain/models#stream)

In [14]:
full = None
for chunk in model.stream("Write two short sentences thanking a customer for their patience."):
    print(chunk.text, end="", flush=True)
    full = chunk if full is None else full + chunk      # chunks add up: text and usage
print("\n\nusage, added up:", full.usage_metadata)

Thank you

 so much for your patience. We truly appreciate your understanding.



usage, added up: {'input_tokens': 12, 'output_tokens': 14, 'total_tokens': 26, 'input_token_details': {'cache_read': 0}}


The text arrives in pieces, as in session 2. Usage is different: Gemini spreads it across the
chunks and other vendors put it on the last one. Chunks add up with `+`, text and usage alike, so
adding them is the habit that works on every vendor. v1 is a batch job over the inbox, so it doesn't
stream; streaming comes back in session 6, where a customer is watching.

### 2.6 Model profiles: capabilities as data

Before you switch to a model, you want to know what it can do. LangChain ships that as data.

**Docs:** [model profiles](https://docs.langchain.com/oss/python/langchain/models#model-profiles) (beta)

In [15]:
for model_id, ok in AVAILABLE.items():
    if not ok:
        print(f"{model_id:38} skipped")
        continue
    profile = init_chat_model(model_id).profile        # data shipped with the package; no call made
    if not profile:                                     # None or {}: nobody has filled it in
        print(f"{model_id:38} no profile")
        continue
    print(f"{model_id:38} context {profile['max_input_tokens']:>9,}  images {profile['image_inputs']!s:5}"
          f"  native structured output {profile['structured_output']}")

print(f"\n{'google_genai:gemini-3.8-flash':38} {init_chat_model('google_genai:gemini-3.8-flash').profile!r}")

google_genai:gemini-3.5-flash-lite     context 1,048,576  images True   native structured output True


groq:openai/gpt-oss-20b                context   131,072  images False  native structured output True


openai:gpt-5.4-nano                    context   400,000  images True   native structured output True


anthropic:claude-haiku-4-5-20251001    context   200,000  images True   native structured output False
ollama:llama3.2:3b                     no profile

google_genai:gemini-3.8-flash          {}


Read a profile as a hint, not a contract. It comes from a public dataset and lags new models
(`gemini-3.8-flash` comes back empty), and "native structured output: False" for Anthropic doesn't
mean it can't do structured output; it means it does it through a tool call (section 3). An empty or
missing profile means "unknown", never "fine".

**Your turn.** Make one call to Groq's `openai/gpt-oss-20b` through `init_chat_model` and keep the
reply. The checker reads its usage and which model answered.

In [ ]:
groq_reply = "groq:openai/gpt-oss-120b"        # <- build a model for groq:openai/gpt-oss-20b and invoke it


def check_turn_1(reply):
    if not os.getenv("GROQ_API_KEY"):
        return "no GROQ_API_KEY yet: section 3.5 has the two-minute setup"
    if reply is None:
        return "build the model, call it, and keep the reply in groq_reply"
    if set(reply.usage_metadata) < {"input_tokens", "output_tokens", "total_tokens"}:
        return "that doesn't look like a model reply"
    return f"passed: {reply.response_metadata.get('model_name')} answered in {reply.usage_metadata['output_tokens']} tokens"


print(check_turn_1(groq_reply))

build the model, call it, and keep the reply in groq_reply


## 3. Five providers, one loop

The point of the interface: the same request to five vendors, in one plain loop. One ticket session 3
found ambiguous (`TCK-003`: a late order and an angry customer), the v1 triage prompt, and
`with_structured_output(TriageResult)` on each.

**Docs:** [Google](https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai) ·
[Groq](https://docs.langchain.com/oss/python/integrations/chat/groq) ·
[OpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai) ·
[Anthropic](https://docs.langchain.com/oss/python/integrations/chat/anthropic) ·
[Ollama](https://docs.langchain.com/oss/python/integrations/chat/ollama) ·
[structured output](https://docs.langchain.com/oss/python/langchain/models#structured-output)

In [17]:
ticket = by_id["TCK-003"]
messages = [("system", v1.triage_system()), ("human", ticket.as_block())]
print(ticket.body, "\n")


def five_providers():
    for model_id, ok in AVAILABLE.items():
        if not ok:
            print(f"{model_id:38} skipped")
            continue
        structured = init_chat_model(model_id).with_structured_output(TriageResult, include_raw=True)
        start = time.perf_counter()
        try:
            out = structured.invoke(messages)
        except Exception as err:          # Groq sometimes rejects its own model's tool call (section 5.1)
            print(f"{model_id:38} failed: {type(err).__name__}")
            continue
        seconds = time.perf_counter() - start
        result, usage = out["parsed"], out["raw"].usage_metadata
        if result is None:
            print(f"{model_id:38} {seconds:4.1f}s  no usable answer: {out['parsing_error']}")
            continue
        print(f"{model_id:38} {seconds:4.1f}s  {result.category:13} {result.urgency:7}"
              f" in {usage['input_tokens']:5}  out {usage['output_tokens']:4}")


five_providers()

THIS IS THE THIRD TIME I AM WRITING AND NOBODY HAS REPLIED. I PAID FOR TWO KETTLES ON THE 1ST AND THE ORDER STILL SAYS PROCESSING. EITHER SHIP THEM TODAY OR REFUND ME. ORD-5002. I WILL BE POSTING ABOUT THIS. 



google_genai:gemini-3.5-flash-lite      1.5s  order_status  high    in   246  out   58


groq:openai/gpt-oss-20b                 0.9s  complaint     high    in   570  out  164


openai:gpt-5.4-nano                     1.3s  order_status  high    in   520  out   55


anthropic:claude-haiku-4-5-20251001     1.8s  order_status  high    in  1193  out  127


ollama:llama3.2:3b                      7.6s  complaint     low     in   261  out   49


Run it again before reading anything into it:

In [18]:
five_providers()

google_genai:gemini-3.5-flash-lite      1.2s  order_status  high    in   246  out   58


groq:openai/gpt-oss-20b                 0.8s  complaint     high    in   570  out  295


openai:gpt-5.4-nano                     1.8s  order_status  high    in   520  out   54


anthropic:claude-haiku-4-5-20251001     1.6s  order_status  high    in  1193  out  128


ollama:llama3.2:3b                      3.1s  complaint     low     in   261  out   57


### 3.1 Read the output

Three vendors say `order_status` and two say `complaint`. Our label is `complaint`: session 3's
convention is that ShopWise ignoring a customer twice makes it a complaint, even when the order is
also late. The two that agree with us are the open-weights models, which is a reminder that a bigger
model doesn't share your house rules just because it's bigger. The local model rated a customer
shouting in capitals as `low` urgency, twice.

The second run matters as much as the first. Groq gave the same label with nearly twice the output
tokens, because it reasons before answering and reasons differently each time, and labels can flip
between runs too (they did while this notebook was being written). So "it gives the same label" is not
an acceptance test. A score over all twenty tickets is (section 5).

### 3.2 The same request, billed differently

Each vendor receives the schema in its own way, and some of them bill you for it. Same messages, with
and without the schema:

In [19]:
for model_id, ok in AVAILABLE.items():
    if not ok:
        continue
    model = init_chat_model(model_id)
    plain = model.invoke(messages).usage_metadata["input_tokens"]
    try:
        out = model.with_structured_output(TriageResult, include_raw=True).invoke(messages)
    except Exception as err:          # Groq sometimes rejects its own model's tool call (section 5.1)
        print(f"{model_id:38} without {plain:5}   with schema: failed, {type(err).__name__}")
        continue
    with_schema = out["raw"].usage_metadata["input_tokens"]
    print(f"{model_id:38} without {plain:5}   with {with_schema:5}   schema adds {with_schema - plain:+5}")

google_genai:gemini-3.5-flash-lite     without   246   with   246   schema adds    +0


groq:openai/gpt-oss-20b                without   306   with schema: failed, BadRequestError


openai:gpt-5.4-nano                    without   242   with   520   schema adds  +278


anthropic:claude-haiku-4-5-20251001    without   269   with  1193   schema adds  +924


ollama:llama3.2:3b                     without   261   with   261   schema adds    +0


Without the schema, all five vendors sit within about 65 tokens of each other. With it, Gemini and
Ollama add nothing, because they apply the schema while generating. OpenAI adds 278 tokens and
Anthropic 924, on every call, and the only place you'd ever see it is the bill. Groq's line failed
on the saved run with the tool-name bug from section 5.1; when it worked, it added 264. The whole
spread is the schema.

| Vendor | How `with_structured_output` sends the schema by default |
|---|---|
| Gemini | as a JSON Schema in the request (`response_json_schema`); the decoder follows it |
| OpenAI | as a strict JSON Schema (`response_format`) |
| Anthropic, Groq | as a **tool** the model is forced to call |
| Ollama | as a JSON Schema (`format`) |

How it travels and what it costs are different questions: don't conclude "tool call = expensive".
The habit is to check on the vendor you use, and to keep schemas small, because some vendors bill
every field description on every call.

### 3.3 What else stays the vendor's

The interface standardises the top layer. Underneath, each vendor still reports in its own words:

In [20]:
for model_id, ok in AVAILABLE.items():
    if not ok:
        continue
    r = init_chat_model(model_id).invoke("Say hi.")
    details = {k: v for k, v in r.usage_metadata.items() if k.endswith("_details")}
    print(f"{model_id:38} model_name={r.response_metadata.get('model_name')!s:28} {details}")

google_genai:gemini-3.5-flash-lite     model_name=gemini-3.5-flash-lite        {'input_token_details': {'cache_read': 0}}


groq:openai/gpt-oss-20b                model_name=openai/gpt-oss-20b           {'output_token_details': {'reasoning': 40}}


openai:gpt-5.4-nano                    model_name=gpt-5.4-nano-2026-03-17      {'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


anthropic:claude-haiku-4-5-20251001    model_name=claude-haiku-4-5-20251001    {'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}


ollama:llama3.2:3b                     model_name=llama3.2:3b                  {}


`response_metadata` is the vendor's own envelope. OpenAI answers with a dated model ID you didn't ask
for, which is the string a price table matches on. The `*_token_details` (cache reads, reasoning) are
reported only where the vendor reports them.

### 3.4 Errors: a standard is arriving, package by package

Ask each provider for a model that doesn't exist:

**Docs:** [model exceptions](https://docs.langchain.com/oss/python/langchain/models#model-exceptions)

In [21]:
from langchain_core.exceptions import ModelError

for model_id, ok in AVAILABLE.items():
    if not ok:
        continue
    provider = model_id.split(":")[0]
    try:
        init_chat_model(f"{provider}:no-such-model", max_retries=1).invoke("hi")
    except Exception as err:
        kind = f"standard, is_retryable={err.is_retryable}" if isinstance(err, ModelError) else "vendor's own"
        print(f"{provider:13} {type(err).__name__:36} {kind}")

google_genai  ChatGoogleGenerativeAIError          vendor's own
groq          NotFoundError                        vendor's own


openai        OpenAIModelNotFoundError             standard, is_retryable=False


anthropic     AnthropicModelNotFoundError          standard, is_retryable=False
ollama        ResponseError                        vendor's own


Five vendors, five exception classes. Since August 2026 (`langchain-core` 1.6.0), LangChain defines
standard ones (`ModelNotFoundError`, `ModelRateLimitError`, `ModelTimeoutError`, ...) with an
`is_retryable` flag, and each integration's class also inherits from the vendor SDK's own, so you can
catch either. OpenAI's and Anthropic's packages raise them already. Google's package added them in
4.4.0, but we pin 4.3.2. Groq and Ollama haven't yet.

That's normal for a moving standard, and it's why you pin versions and read the changelog before you
bump one (appendix F). It matters in section 5, where "which errors should switch vendors?" depends on
telling a 429 from a 400.

<img src="figures/where-it-stops.png" width="900"
     alt="Where the standard interface stops, as two layers. The standard layer on top: messages,
     .text, the three usage_metadata keys, with_structured_output, and the new exception types where
     an integration has adopted them. The vendor layer underneath: how the schema travels (decoder,
     strict format or forced tool call) and what it adds to the bill, the token-detail sub-dicts,
     response_metadata with dated model IDs, vendor knobs such as thinking_level and reasoning_effort,
     and model profiles that can be missing.">

### 3.5 Setting up Groq (free, two minutes)

1. Sign in at <https://console.groq.com> with Google or GitHub.
2. **API Keys → Create API Key**, copy it.
3. Add `GROQ_API_KEY=gsk_...` to `.env`.
4. **Restart the kernel.** `.env` is read when `config` is imported, so a running kernel can't
   see a key you added afterwards.

Groq runs open-weights models on its own chips, and fast. Its free plan for `openai/gpt-oss-20b`
allows 30 requests and 8,000 tokens a minute. The token limit is the one you'll hit.

### 3.6 Setting up Ollama (free, local, optional)

1. Install it from <https://ollama.com/download> and start it.
2. `ollama pull llama3.2:3b` (about 2 GB, once).
3. Nothing to add to `.env`: no key, no network, no bill.

A 3B model on a laptop is much weaker than the hosted ones, as you'll see in section 4. It's here
because it's the honest baseline for "runs anywhere, sends nothing to a vendor", and sessions 17–18
fine-tune a model and serve it with Ollama.

## 4. Structured output, and the check the framework can't do

The second half of the interface, and the most important ten minutes tonight.

### 4.1 Triage ports in one line

`with_structured_output(TriageResult)` replaces v0.2's `response_schema` plumbing:

In [22]:
model = init_chat_model(settings.model_id)
triage_model = model.with_structured_output(TriageResult)
print(triage_model.invoke([("system", v1.triage_system()), ("human", by_id["TCK-007"].as_block())]))

category='refund' urgency='medium' sentiment='calm' needs_human=True summary='Customer asking about the processing time for refund reference REF-9001.'


`TriageResult`'s validators still run (the summary is tidied, money still means `needs_human=True`),
because they need nothing from outside the object. And the schema file didn't change at all in the
port:

In [23]:
for name in ["schemas.py", "policy.py", "prompts.yml"]:
    old = hashlib.sha256(Path("../session-03/project", name).read_bytes()).hexdigest()[:12]
    new = hashlib.sha256(Path("project", name).read_bytes()).hexdigest()[:12]
    print(f"{name:12} v0.2 {old}   v1 {new}   {'identical' if old == new else 'CHANGED'}")

schemas.py   v0.2 c9c35e7e69e6   v1 c9c35e7e69e6   identical
policy.py    v0.2 d20e91fc8e98   v1 d20e91fc8e98   identical
prompts.yml  v0.2 55b57853fa90   v1 55b57853fa90   identical


### 4.2 The obvious port of `extract()`

`TicketFacts` should port the same way. Here's a realistic ticket: support tools often paste the
customer's account summary under the message, and this one mentions an order the customer didn't.

In [24]:
kettle = Ticket.model_validate({
    "id": "TCK-901", "subject": "Kettle stopped working",
    "body": "My BrewMaster kettle stopped heating after a week. What can I do?",
    "from": "rahim@example.com", "category": "warranty", "received": "2026-08-20"})
account = "Account summary: gold tier, customer since 2024, last order ORD-5004 (delivered 2026-07-20)."

decoy_messages = [("system", v1.PROMPTS["facts_prompt"]),
                  ("human", kettle.as_block() + "\n\n" + account)]
ticket_context = {"ticket": f"{kettle.subject}\n{kettle.body}"}
print(decoy_messages[1][1])

<ticket id="TCK-901">
Subject: Kettle stopped working
My BrewMaster kettle stopped heating after a week. What can I do?
</ticket>

Account summary: gold tier, customer since 2024, last order ORD-5004 (delivered 2026-07-20).


`facts_prompt` says "copy only what the customer actually wrote". `ORD-5004` is in the account
summary, not in the customer's words: maybe the right order, maybe not (a gold customer has several).
Session 3 put a check in `TicketFacts` for exactly this. Port `extract()` the obvious way and try it on
the two free models and OpenAI's smallest:

In [25]:
for model_id in ["groq:openai/gpt-oss-20b", "ollama:llama3.2:3b", "openai:gpt-5.4-nano"]:
    if not AVAILABLE[model_id]:
        print(f"{model_id:30} skipped")
        continue
    naive = init_chat_model(model_id).with_structured_output(TicketFacts)
    runs = [naive.invoke(decoy_messages).order_ids for _ in range(3)]
    print(f"{model_id:30} {runs}")

groq:openai/gpt-oss-20b        [['ORD-5004'], [], []]


ollama:llama3.2:3b             [['ORD-5004'], ['ORD-5004'], ['ORD-5004']]


openai:gpt-5.4-nano            [['ORD-5004'], ['ORD-5004'], ['ORD-5004']]


`ORD-5004` comes back as a valid `TicketFacts`, and nothing objects. The counts move between runs
(Groq copied it in two runs of three while this was being written, none in the saved run), but the
local model and OpenAI's nano copy it nearly every time. The check is still in the class. It just
never ran. You don't need a model to see why:

In [26]:
from langchain_core.output_parsers import PydanticOutputParser

answer = '{"order_ids": ["ORD-5004"], "product_kind": "kitchen", "days_since_delivery": null, "opened": null}'

parser = PydanticOutputParser(pydantic_object=TicketFacts)    # the kind of parser with_structured_output uses
print("through the framework's parser:", parser.parse(answer))

try:
    TicketFacts.model_validate_json(answer, context=ticket_context)
except ValidationError as err:
    print("with the ticket as context:    ", err.errors()[0]["msg"])

through the framework's parser: order_ids=['ORD-5004'] product_kind='kitchen' days_since_delivery=None opened=None
with the ticket as context:     Value error, ORD-5004 is not in the ticket


### 4.3 Why: the framework has nowhere to put your context

`must_appear_in_ticket` needs the ticket text, and session 3 passed it as **validation context**:
`model_validate_json(text, context={"ticket": ...})`. No parser in LangChain has an argument for that.
They call `model_validate(obj)`, or `Schema(**args)` for tool calls, or the OpenAI SDK validates for
them. With no context, the validator does what it was written to do: skip the check.

The framework kept its promise. The promise just didn't include your context. **It fails at nothing,
which is worse than failing.** In session 5 the assistant looks orders up, and an order number nobody
checked means reading somebody else's order.

`include_raw=True` doesn't rescue you either: when validation fails, Gemini returns an
`OutputParserException` in `parsing_error`, Anthropic the bare `ValidationError`, and OpenAI raises it
straight out of `.invoke()`. Three behaviours behind one interface.

### 4.4 Keep the seam

So v1 keeps session 3's `ask_json`, moved onto LangChain (open `project/utils.py`; it's about a dozen
lines):

1. ask for a **dict** schema, `schema.model_json_schema()`, so every provider hands back a plain dict;
2. validate it in our code, `schema.model_validate(parsed, context=context)`;
3. if that fails, ask once more with the reason attached, then give up.

It also asks every provider for JSON in the reply (`method="json_schema"`) rather than a forced tool
call. Section 5 tells the story of why.

In [27]:
for model_id in ["groq:openai/gpt-oss-20b", "ollama:llama3.2:3b", "openai:gpt-5.4-nano",
                 "google_genai:gemini-3.5-flash-lite"]:
    if not AVAILABLE[model_id]:
        print(f"{model_id:36} skipped")
        continue
    try:
        facts, _ = ask_json(init_chat_model(model_id), TicketFacts, decoy_messages, context=ticket_context)
        print(f"{model_id:36} {facts.order_ids}")
    except ValidationError as err:
        print(f"{model_id:36} rejected twice, never reaches the next step: {err.errors()[0]['msg']}")

groq:openai/gpt-oss-20b              []


ollama:llama3.2:3b                   []


openai:gpt-5.4-nano                  []


google_genai:gemini-3.5-flash-lite   []


Every model now comes back with no order number. The ones that copied `ORD-5004` had their answer
rejected by `must_appear_in_ticket`, were told why, and corrected themselves on the second attempt.
In LangSmith (section 7) you can see both attempts side by side.

The trade-off, stated: with a dict schema, OpenAI runs in its non-strict mode. That's fine, because
our validator is the real check.

| Absorbed by the framework | Still yours |
|---|---|
| the client, timeouts and retries | checking the answer against the source (validation context) |
| each vendor's schema dialect | the repair attempt |
| the message format | every decision in `policy.py` |
| token counts under one set of names | choosing *how* the schema travels (`method=`) |

<img src="figures/the-seam.png" width="900"
     alt="The seam, as two paths from a model to a TicketFacts object. The naive path: the model's
     answer goes through the framework's parser, validation context is lost, the order-number check is
     skipped and an invented ORD-5004 comes out as valid. The kept path: a dict schema goes out, a
     plain dict comes back on every vendor, our own model_validate runs with the ticket as context,
     the check rejects ORD-5004, one repair attempt is made with the reason attached, and the corrected
     answer has no order number.">

**Your turn.** Write `extract_facts(model, ticket, extra="")`: it sends the ticket (plus any `extra`
text, like the account summary) with `facts_prompt`, and returns the `TicketFacts` from `ask_json`.
The checker uses a stand-in model that copies the account summary's order number on its first
attempt, so only a version that passes the ticket as context comes back clean. No key needed.

In [28]:
def extract_facts(model, ticket, extra=""):
    messages = [("system", v1.PROMPTS["facts_prompt"]), ("human", ticket.as_block() + extra)]
    return None          # <- call ask_json and return just the TicketFacts


def check_turn_2():
    from langchain_core.runnables import RunnableLambda

    class CopiesTheAccountSummary:
        """A stand-in model: invents ORD-5004 first, corrects itself when told why."""
        def with_structured_output(self, schema, **kwargs):
            return RunnableLambda(self.answer)

        def answer(self, messages):
            told = any("rejected" in str(m) for m in messages)
            parsed = {"order_ids": [] if told else ["ORD-5004"], "product_kind": "kitchen",
                      "days_since_delivery": None, "opened": None}
            usage = {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}
            return {"parsed": parsed, "raw": AIMessage("", usage_metadata=usage), "parsing_error": None}

    facts = extract_facts(CopiesTheAccountSummary(), kettle, extra="\n\n" + account)
    if facts is None:
        return "extract_facts returned None"
    if facts.order_ids:
        return f"invented {facts.order_ids} got through: is the ticket text going in as context?"
    return "passed: the invented order number was caught and repaired"


print(check_turn_2())

extract_facts returned None


## 5. Plan B: switch or fall back

"A Plan B vendor" means two different things, and a reviewer will ask which one you built.

| | **Switch** | **Fallback** |
|---|---|---|
| when | planned: a price rise, a model being retired | unplanned: an outage, a spent quota, a timeout |
| who decides | a person, in a deploy | the code, per call, automatically |
| what changes | `SHOPWISE_PROVIDER` and `SHOPWISE_MODEL` in `.env` | nothing: it's already attached |
| scored first | everything, it's a new deploy | the fallback model, once, before you trust it |

Both are real. On 29 September 2026 Anthropic's API returned errors for about two hours in the afternoon (UTC) and
OpenAI's for over five hours that evening: a fallback between them would have worked that day. And
vendor switches mostly happen for cost or quality, planned weeks ahead.

**Docs:** [`with_fallbacks`](https://reference.langchain.com/python/langchain-core/runnables/base/Runnable/with_fallbacks)

### 5.1 The switch: one string, then a score

Switching is one line in `.env`. Knowing whether you should is `score.py` (beside this notebook),
which now takes any provider string.

When we first scored Groq for this notebook, **9 of 20 tickets failed** with `BadRequestError`. The
model was asked to call a tool named `TriageResult` and called `triageResult`, and Groq's server
refused it. That had nothing to do with the model's judgement, and no demo would have found it. That's
why `ask_json` asks for `method="json_schema"`. With that fixed, both scored (this takes a few
minutes on free keys):

In [29]:
for provider in [settings.model_id, settings.FALLBACK]:
    if not AVAILABLE.get(provider, True):
        print(f"{provider}: skipped\n")
        continue
    result = subprocess.run([sys.executable, "score.py", "--provider", provider],
                            capture_output=True, text=True)
    print(result.stdout or result.stderr[-500:])

scoring google_genai:gemini-3.5-flash-lite on 20 tickets, 1 run(s)

run 1: triage 17/20 · 1109 output tokens · order numbers 7 found, 0 missed, 0 invented, 0 rejected
    TCK-003  labelled complaint         model said order_status
    TCK-015  labelled warranty          model said product_question
    TCK-016  labelled refund            model said order_status



scoring groq:openai/gpt-oss-20b on 20 tickets, 1 run(s)

run 1: triage 17/20 · 4788 output tokens · order numbers 7 found, 0 missed, 0 invented, 0 rejected
    TCK-002  labelled refund            model said complaint
    TCK-012  labelled shipping          model said other
    TCK-015  labelled warranty          model said product_question



Gemini 17/20 and Groq 17/20 on the saved run: the same band (session 3 measured 16–19 for these
rules), with different misses, and only `TCK-015` shared. The bigger difference is the output
tokens: Groq used more than four times as many (about 4,800 against 1,100), because `gpt-oss-20b`
reasons before it answers. On a paid plan that's the bill. On Groq's free plan it's the
8,000-tokens-a-minute limit, which is why its scoring run is the slower one.

### 5.2 The fallback

`with_fallbacks` wraps one model with a list of others. Simulate an outage with a model ID that
doesn't exist:

In [30]:
broken = init_chat_model("google_genai:gemini-0-does-not-exist", max_retries=1)
backup = init_chat_model(settings.FALLBACK)
with_plan_b = broken.with_fallbacks([backup])

start = time.perf_counter()
reply = with_plan_b.invoke("In one sentence: where do I find my order number?")
print(f"{time.perf_counter() - start:.1f}s, answered by {reply.response_metadata.get('model_name')}")
print(reply.text)

1.9s, answered by openai/gpt-oss-20b
Check the confirmation email, your account’s order‑history page, or the receipt – the order number is listed in one of those places.


v1's `build_model()` does exactly this with `settings.FALLBACK`, and `with_structured_output` reaches
both models through it. Before you ship a fallback, a reviewer asks four questions.

**1. What should fall back?** Look at what it catches:

In [31]:
print(with_plan_b.exceptions_to_handle)

(<class 'Exception'>,)


Everything. A 429 or a 503 should fall back. Your own malformed request (a 400) shouldn't: Groq might
accept it and hide your bug. With the standard exceptions that's one line,
`exceptions_to_handle=(ModelRateLimitError, ModelAPIError, ModelTimeoutError, ModelConnectionError)`.
But our pinned Gemini package raises one class for 400 and 429 alike (section 3.4), so v1 keeps the
default for now; the ⭐⭐⭐ exercise writes the precise version. A failed *validation* never triggers
the fallback, by placement: `ask_json` validates after the call has returned.

**2. How long before it falls back?** Retries run first. A 404 isn't retried, which is why the
fallback above answered within a second. A 429 is: with `max_retries=4`, the waits of roughly 1, 2 and 4 seconds (plus
jitter) all happen before Groq is asked, and LangChain's default of 6 attempts means about 30 seconds.
With a fallback attached, fewer retries on the primary is usually the better trade.

**3. Can the fallback do the job?**

In [32]:
for model_id in [settings.model_id, settings.FALLBACK]:
    profile = init_chat_model(model_id).profile
    print(f"{model_id:38} images {profile['image_inputs']!s:5}  context {profile['max_input_tokens']:>9,}")

google_genai:gemini-3.5-flash-lite     images True   context 1,048,576
groq:openai/gpt-oss-20b                images False  context   131,072


No image input on Groq's model: `TCK-011`'s photo (session 3) can't fall back. And a context window an
eighth of Gemini's: a long prompt that works on the primary can fail on the fallback.

**4. Is it as good?** Section 5.1's score. And when it does take over, you want to know: every
fallback call shows up in LangSmith (section 7) with the model that actually answered. A fallback
nobody notices is an outage nobody reported.

Later: `ModelFallbackMiddleware` and `ModelRetryMiddleware` do the same inside an agent (session 7),
and gateways do it at the network layer.

<img src="figures/plan-b.png" width="900"
     alt="Plan B in two halves. Left, the switch: a person changes the provider string in .env, scores
     the new model on the twenty labelled tickets, then deploys. Right, the fallback: a call to Gemini
     fails, retries run with backoff of about one, two and four seconds, then the same request goes to
     Groq, which answers. Below, the four reviewer questions: what should fall back (429 and 5xx yes,
     400 no), how long before it does, can the fallback do the job (no images, smaller context), and
     is it as good (score it, watch who answered).">

**Your turn.** Attach `settings.FALLBACK` to the broken model below and get an answer. The checker
reads which model replied.

In [33]:
primary = init_chat_model("google_genai:gemini-0-does-not-exist", max_retries=1)
plan_b = primary          # <- attach the fallback

try:
    turn_3_reply = plan_b.invoke("Say hello in three words.")
except Exception as err:
    turn_3_reply = err


def check_turn_3(reply):
    if not os.getenv("GROQ_API_KEY"):
        return "no GROQ_API_KEY yet: section 3.5"
    if isinstance(reply, Exception):
        return f"the primary's error reached you ({type(reply).__name__}): nothing caught it"
    return f"passed: answered by {reply.response_metadata.get('model_name')}"


print(check_turn_3(turn_3_reply))

the primary's error reached you (ChatGoogleGenerativeAIError): nothing caught it


## 6. Many calls: batching, rate limits, caching

v1 is a batch job: 20 tickets, three calls each. Session 2 promised a better way than a loop, and
session 3 promised caching through the framework.

**Docs:** [batch](https://docs.langchain.com/oss/python/langchain/models#batch) ·
[rate limiting](https://docs.langchain.com/oss/python/langchain/models#rate-limiting) ·
[prompt caching](https://docs.langchain.com/oss/python/langchain/models#prompt-caching)

### 6.1 Batching

A model has `.batch()` of its own: `model.batch(["question one", "question two"])` sends both at
once and returns the replies in order. One ticket is three calls, though, so we wrap the function
that handles a ticket in `RunnableLambda`, which gives any function the same `.batch()`.

In [34]:
from langchain_core.runnables import RunnableLambda

model = init_chat_model(settings.model_id)
six = TICKETS[:6]


def triage_one(ticket):
    result, _ = v1.triage(model, ticket)
    return result.category


start = time.perf_counter()
one_by_one = [triage_one(t) for t in six]
print(f"one at a time {time.perf_counter() - start:5.1f}s  {one_by_one}")

start = time.perf_counter()
together = RunnableLambda(triage_one).batch(six, config={"max_concurrency": 6})
print(f"batched       {time.perf_counter() - start:5.1f}s  {together}")

one at a time   6.5s  ['order_status', 'refund', 'order_status', 'order_status', 'return', 'warranty']


batched         1.4s  ['order_status', 'refund', 'order_status', 'order_status', 'warranty', 'warranty']


The calls overlap, so the wall-clock time drops. `triage.py` runs each ticket's whole pipeline this
way, with `return_exceptions=True` so one failed ticket comes back as its error instead of stopping
the other nineteen, and with each ticket's run named after its ID (you'll see why in section 7).

### 6.2 Rate limits

Free tiers allow a handful of requests a minute: Groq's free `gpt-oss-20b` 30 requests and 8,000
tokens a minute; Gemini's limits are per Google Cloud project and shown in AI Studio. Sixty calls at
once is how you meet session 2's 429 for real, and the retries hide it behind long waits. A rate
limiter makes each call wait for a free slot instead:

In [35]:
from langchain_core.rate_limiters import InMemoryRateLimiter

limited = init_chat_model(settings.model_id, rate_limiter=InMemoryRateLimiter(requests_per_second=1))


def say(n):
    return limited.invoke(f"Reply with the number {n} and nothing else.").text


start = time.perf_counter()
print(RunnableLambda(say).batch([1, 2, 3, 4, 5], config={"max_concurrency": 5}),
      f"{time.perf_counter() - start:.1f}s")

['1', '2', '3', '4', '5'] 5.9s


Five calls asked for at once, released about one a second. Three limits to know: it only limits this
one Python process; it counts requests, not tokens (Groq's binding limit is tokens); and each provider
needs its own, which is why `make_model()` in `utils.py` gives every model its own limiter. v1's
defaults, `MAX_CONCURRENCY=4` and `REQUESTS_PER_SECOND=0.25`, are safe on free keys and slow: the whole
inbox takes about four minutes. On a billed key you'd raise them (⭐⭐ exercise).

### 6.3 Caching, read through the interface

Session 3 measured Gemini's implicit cache with Google's SDK on a 25,000-token policy library. The
same library through `init_chat_model`:

In [36]:
policy = Path("../session-02/returns_policy.md").read_text(encoding="utf-8")
partner = policy.replace("ShopWise", "Northwind").replace("15 days", "10 days").replace("30 days", "45 days")
documents = [f"<document index='{i}'>\n{partner}\n</document>" for i in range(60)]
documents.insert(37, f"<document index='shopwise'>\n{policy}\n</document>")
LIBRARY = "\n".join(documents)

for question in ["How many days do I have to return opened headphones?",
                 "Who pays for return shipping?",
                 "How do I start a return?"]:
    usage = model.invoke([("system", LIBRARY),
                          ("human", f"Under ShopWise's policy: {question} One sentence.")]).usage_metadata
    print(f"input {usage['input_tokens']:6,}   cache_read {usage['input_token_details'].get('cache_read', 0):6,}")

input 24,655   cache_read      0


input 24,650   cache_read 20,449


input 24,651   cache_read 20,449


The first call reads nothing from the cache. From the second, about 20,450 of the ~24,650 input
tokens come from it: the same number session 3 measured with Google's own SDK, now reported under
`input_token_details["cache_read"]`, the key every vendor uses in LangChain.

| Vendor | How caching works | Through LangChain |
|---|---|---|
| Gemini | implicit, automatic; explicit with a cache you create (💳 on Flash-Lite) | automatic; `cached_content=` for explicit |
| OpenAI | automatic from 1,024 tokens | automatic |
| Anthropic | you mark where the cached part ends | `cache_control` on a content block, or `AnthropicPromptCachingMiddleware` |

The mechanism stays each vendor's. The reading is standard: `cache_read` on every vendor, plus
`cache_creation` on Anthropic. In section 7 a cached call is visibly cheaper: LangSmith prices cache
reads on this model at a tenth of normal input.

## 7. LangSmith: two environment variables, zero code changes

Until now, "what did the model actually receive?" meant adding a `print` and running it again.
LangSmith records every call LangChain makes (the prompt as sent, the reply, latency, tokens and
cost) and you switch it on without touching the code. Tonight is a first look: set it up, see that
it's logging, and learn your way around. We'll do more with it in later sessions.

**Docs:** [create an account and API key](https://docs.langchain.com/langsmith/create-account-api-key) ·
[tracing quickstart](https://docs.langchain.com/langsmith/observability-quickstart) ·
[LangSmith with LangChain](https://docs.langchain.com/oss/python/langchain/observability) ·
[concepts](https://docs.langchain.com/langsmith/observability-concepts) ·
[cost tracking](https://docs.langchain.com/langsmith/cost-tracking)

### 7.1 Set it up

1. Sign up at <https://smith.langchain.com> (Google or GitHub is fine).
2. **Settings → API Keys → Create API Key**. Copy it; it starts with `lsv2_`.
3. Add three lines to `.env`:

   ```
   LANGSMITH_TRACING=true
   LANGSMITH_API_KEY=lsv2_...
   LANGSMITH_PROJECT=shopwise
   ```

4. **Restart the kernel**, then run the setup cell again.

The names have no `SHOPWISE_` prefix because LangSmith's own SDK reads them. The project is just a
folder for runs; it's created the first time something is logged to it. The free Developer plan
allows 5,000 traces a month, kept for 14 days.

In [37]:
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"), "| project:", os.getenv("LANGSMITH_PROJECT"))

TRACING_CODE = ["import langsmith", "from langsmith", "@traceable", "LangChainTracer", "callbacks="]
found = [f"{path.name}: {word}" for path in Path("project").glob("*.py")
         for word in TRACING_CODE if word in path.read_text()]
print("tracing code in project/:", found or "none")

LANGSMITH_TRACING: true | project: shopwise-cse
tracing code in project/: none


No tracing code anywhere in the project. Every LangChain call checks those variables and, if tracing
is on, sends a copy of what it did to LangSmith in the background.

### 7.2 It's already logging

Open <https://smith.langchain.com>, then **Projects → shopwise**. Every call in this notebook since
you set the variables is already there. Run the project too (`uv run python triage.py --limit 3`
from `project/`, or section 9's cell) and come back.

### 7.3 A tour of the interface

**The run list.** One row per top-level run, newest first. `triage.py` names each ticket's run after
its ID, so you see `TCK-001`, `TCK-002`, ... with latency, tokens, cost and status.

<img src="figures/trace-list.png" width="900" alt="The LangSmith run list for the shopwise project:
one row per ticket, TCK-001 to TCK-003, with columns for status, latency, tokens and cost.">

**One run.** Click `TCK-001`. On the left, the tree of everything that ticket did: `TriageResult`,
`TicketFacts` and `draft_reply`, each opening onto the model call inside it. Click a model call to
see, on the right:

| Look at | Why it's useful |
|---|---|
| **Input** | the messages exactly as sent: system prompt, ticket, everything spliced in. The first thing to check when an answer surprises you |
| **Output** | what came back, before your code touched it |
| **Latency** | per step, so you can see which call was slow |
| **Tokens and cost** | per call, and added up for the whole ticket |
| **Metadata** | the provider and model that actually answered, so a fallback call stands out |

<img src="figures/trace-detail.png" width="900" alt="One ticket's run opened in LangSmith. The tree on
the left shows TriageResult, TicketFacts and draft_reply, each containing a ChatGoogleGenerativeAI
model call. The panel on the right shows the selected call's input messages, its output, latency,
token counts and cost.">

The rows named `RunnableSequence`, `RunnableParallel` and so on are LangChain's plumbing:
`with_structured_output` is a small pipeline of a model call and a parser. The model calls are the
rows that matter, and the **LLM** filter shows only those.

<img src="figures/anatomy-of-a-trace.png" width="900"
     alt="Anatomy of one ticket's trace. The root run TCK-001 contains three steps: TriageResult,
     TicketFacts and draft_reply. Each step contains one model call with its input messages, output,
     latency, input and output tokens and cost; the costs add up at the root. A side panel shows a
     fallback: a failed Gemini call followed by a Groq call that answered, with the provider recorded
     in its metadata.">

### 7.4 Cost, priced for you

LangSmith prices each call from a price table it keeps per model: for `gemini-3.5-flash-lite`, $0.30
per million input tokens and $2.50 per million output, and cached input at $0.03. One ticket through v1 (three calls, about 1,200 tokens in and 200 out) comes to $0.0008–0.0009,
so the whole inbox costs under two cents.
Two things to know:

- These are **list prices**. On Gemini's free tier your bill is zero; the column shows what the same
  traffic would cost on a paid key, which is the number you need before you launch.
- A model with no price entry shows no cost: Ollama, correctly, since it runs on your laptop. A model
  that's priced later isn't re-priced backwards.

### 7.5 One habit

When an answer looks wrong, open its run before you add a `print`. It's already there, it shows
exactly what the model saw, and it covers runs that finished hours ago. Filter by **Error** first.

### 7.6 Later in the course

Tagging runs and searching them by ticket ID, keeping customer data out of traces, recording which
prompt wording produced a run, and comparing two prompts as experiments: session 9, where the
assistant gets a proper evaluation set. A hosted prompt registry: session 16.

## 8. The ledger: what the port bought

Count it rather than assert it. Code lines only (no blanks, comments or docstrings), and the
Google-shaped lines from the opening, in v0.2 and in v1:

In [38]:
def code_lines(path):
    """Lines of code: not blank, not a comment, not inside a docstring."""
    count, in_docstring = 0, False
    for line in path.read_text().splitlines():
        stripped = line.strip()
        if stripped.count('"""') == 1:          # a docstring opens or closes here
            in_docstring = not in_docstring
            continue
        if in_docstring or not stripped or stripped.startswith(("#", '"""')):
            continue
        count += 1
    return count


def google_lines(path):
    return sum(1 for line in path.read_text().splitlines()
               if any(word in line.split("#")[0] for word in GOOGLE_SHAPED))


for version_name, folder in [("v0.2", Path("../session-03/project")), ("v1", Path("project"))]:
    for name in ["utils.py", "triage.py"]:
        path = folder / name
        print(f"{version_name:5} {name:10} {code_lines(path):4} code lines   {google_lines(path):3} Google-shaped")

v0.2  utils.py     43 code lines    13 Google-shaped
v0.2  triage.py   105 code lines     8 Google-shaped
v1    utils.py     56 code lines     1 Google-shaped
v1    triage.py   100 code lines     0 Google-shaped


| | v0.2 | v1 |
|---|---|---|
| code lines, `utils.py` + `triage.py` | 148 | 156 |
| Google-shaped lines | 21 | 1, and it's the string `"google_genai"` in the table of key names |

The port made the code **longer**, not shorter. What it removed (the hand-built client, `HttpOptions`
in milliseconds, the `ClientError`/`ServerError` pair, the config objects) is smaller than what v1
added: the fallback, the rate limiter, the key check, and an `ask_json` that still does its own
validation, because that was the part the framework couldn't carry. The win isn't length. It's that
the vendor went from being in every function to being one string in `.env`, and that the program now
has a Plan B and a record of every call. What got vaguer: the errors, until the standard exceptions
reach every package.

**The framework debate (ten minutes, not a vote).** Split the room.

- **Team A: raw SDKs, or just the OpenAI-compatible endpoint.** The check that silently stopped
  running in section 4. Exceptions that aren't standard yet. A schema that costs hundreds of tokens on
  some vendors without telling you. A vocabulary that keeps moving.
- **Team B: the framework.** Five integrations nobody here wrote. A fallback in one line. Tracing for
  two environment variables. And sessions 5–12 (tools, agents, graphs, memory) built on the same
  interface.

Land on section 1's criterion, then the tiebreak: **how many models will this program ever call, and
who is on call at 2 a.m.?**

**Three interview questions** (sketch answers in the exercises' solutions):

1. Every vendor offers an OpenAI-compatible endpoint. What does `init_chat_model` buy you beyond it?
2. After moving to LangChain, extraction started letting invented order numbers through, and nothing
   failed. What happened, and where should that check live?
3. You're asked to move to a cheaper model next week. What do you do before changing the string?

## 9. Project v1

<img src="figures/v1-pipeline.png" width="900"
     alt="The v1 pipeline. Settings read PROVIDER, MODEL and FALLBACK from .env and build one provider
     string; build_model creates the model with a rate limiter and attaches the Groq fallback. The
     inbox runs through a batch, several tickets at once. Each ticket goes through triage and
     extraction via ask_json, where our own validation with the ticket as context and one repair
     attempt sit between the model and the result; return_window decides in code; draft_reply writes
     the answer. Every call is recorded in LangSmith alongside.">

| File | What changed | Why it's separate |
|---|---|---|
| `config.py` | `PROVIDER`, `model_id`, `FALLBACK`, `MAX_CONCURRENCY`, `REQUESTS_PER_SECOND`, the LangSmith switches | everything the program can be told, in one typed place |
| `utils.py` | `make_model()`, `build_model()` with the fallback, `ask_json()` on a dict schema; no Google imports | the plumbing every call goes through, and nothing about tickets |
| `triage.py` | `handle_ticket()` for one ticket; `main()` batches the inbox | the assistant's steps, in order |
| `schemas.py`, `policy.py`, `prompts.yml` | unchanged, byte for byte | the contract, the rules and the wording belong to no vendor |
| `../score.py` | `--provider` | measures the assistant; not part of it |

Walk through the files in your editor. In a terminal, from `session-04/project/`:

```bash
uv run python config.py                 # what it resolved to
uv run python triage.py --limit 3       # three tickets, printed
SHOPWISE_MODEL=gemini-0-nope uv run python triage.py --limit 1   # watch the fallback answer
```

In [39]:
result = subprocess.run([sys.executable, "triage.py", "--limit", "3"],
                        cwd="project", capture_output=True, text=True)
print(result.stdout or result.stderr[-800:])

ShopWise Support Assistant v1.0.0 · google_genai:gemini-3.5-flash-lite · fallback groq:openai/gpt-oss-20b · tracing to shopwise-cse · 3 ticket(s)

TCK-001  order_status | medium | calm | needs_human=False
  Customer is asking for tracking update on order ORD-5001.
  orders=['ORD-5001']  days=None  return window: unknown
----------------------------------------------------------------------------
Hello,

I'm sorry your order hasn't arrived yet. Waiting on a package when the tracking stays still is stressful, and I understand why you want to know when it will reach you.

I don't have access to live delivery systems, so I've passed your order details to a colleague on our shipping team. They will look into the delivery status and reply to this email with an update.

ShopWise Support

TCK-002  refund | medium | frustrated | needs_human=True
  Customer requests refund because the left earcup crackles.
  orders=-  days=9  return window: inside
------------------------------------------------

## 10. Wrap-up

### 10.1 The docs habit

Every section tonight linked the page it was taught from. When you meet something new: find its page,
run the smallest example on it, then change one thing. docs.langchain.com also publishes an
[`llms.txt`](https://docs.langchain.com/llms.txt) index, so a coding assistant can read the current
docs instead of what it remembers from 2023.

### 10.2 The card: LangChain v1 versus the internet

| You'll find online (pre-v1) | What we use (v1) |
|---|---|
| `ChatGoogleGenerativeAI(model=...)` everywhere | `init_chat_model("google_genai:...")` |
| `from langchain_community.chat_models import ...` | the vendor's own package, `langchain-google-genai` |
| `LLMChain`, `prompt \| model \| parser` as the main idea | plain function calls; graphs in session 10 |
| `AgentExecutor`, `initialize_agent` | `create_agent` (session 5) |
| `reply.text()`, `reply.content.strip()` | `reply.text`; `content_blocks` |
| `from langchain_core.messages import ...` | `from langchain.messages import ...` (same classes) |
| `langchain.debug = True`, prints | LangSmith |

### 10.3 v0.2 → v1

| v0.2 | v1 |
|---|---|
| 22 lines written for Google's SDK | one provider string in `.env` |
| A Plan B vendor meant a rewrite | a switch is one string and a score; a fallback is one line |
| `ask_json` on Google's `response_schema` | `ask_json` on a dict schema: the same check and repair on every vendor |
| `HttpRetryOptions` built by hand | `max_retries=`, and a fallback when retries run out |
| tickets one at a time | `.batch()`, under a rate limiter |
| cache hits read from Google's own fields | `cache_read`, the same key on three vendors |
| `print`, and run it again | open the run in LangSmith |
| cost: nobody looked | priced per call and per ticket |

### 10.4 Ayesha's order

`TCK-001` is the simplest ticket in the inbox:

In [40]:
backend = yaml.safe_load((settings.DATA_DIR / "backend.yml").read_text(encoding="utf-8"))
print("TCK-001:", by_id["TCK-001"].body)
print("\nwhat v1 extracted: order_ids=['ORD-5001'], checked against the ticket")
print("\nwhat ShopWise's records say:")
print("  ORD-5001", backend["orders"]["ORD-5001"])
print("  REF-9001", backend["refunds"]["REF-9001"])

TCK-001: Hi, I ordered the AeroSound Pro headphones on the 14th and the tracking hasn't moved. Order ORD-5001. Can you tell me when it will arrive?

what v1 extracted: order_ids=['ORD-5001'], checked against the ticket

what ShopWise's records say:
  ORD-5001 {'customer_id': 'C-1001', 'items': [{'sku': 'SKU-100', 'qty': 1}], 'total': 129.0, 'status': 'delivered', 'order_date': '2026-07-14', 'delivered_date': '2026-07-19'}
  REF-9001 {'order_id': 'ORD-5001', 'amount': 129.0, 'status': 'requested', 'reason': 'left earcup crackles', 'requested_date': '2026-08-02'}


v1 did its job: it triaged the ticket, pulled out `ORD-5001`, checked that the number really is in
Ayesha's message, and wrote a polite reply saying a colleague will look at the tracking (section 9's
output). But the order was **delivered on 19 July**, two weeks before she wrote "the tracking hasn't
moved", and the shop already has a refund request from her on the same order.

The order number is verified against the ticket. Nothing is verified against the order. The assistant
has a brain and no hands. Session 5 gives it hands.

**Before then:** work through `exercises.ipynb`. The ⭐⭐⭐ ones join tonight's ideas together: a
fallback that doesn't hide your own bugs, and the same request billed by three vendors.

---

# Appendix

Optional, and not needed for session 5.

## A. Content blocks: reasoning, and a photo

Section 2.4 showed reasoning as a content block. Images go the other way, as a block in the message.
Session 3 sent `TCK-011`'s photo through Google's SDK; here it is through the interface:

In [41]:
import base64

photo = base64.b64encode((settings.DATA_DIR / "images" / "ticket-photo-headphones.jpg").read_bytes()).decode()
message = HumanMessage(content=[
    {"type": "text", "text": by_id["TCK-011"].as_block() +
     "\nThe photo is the one the customer attached. Which product does it show? Two words."},
    {"type": "image", "base64": photo, "mime_type": "image/jpeg"},
])
r = init_chat_model(settings.model_id).invoke([message])
print(r.text, "|", r.usage_metadata["input_tokens"], "input tokens")

Black headphones | 1142 input tokens


The same message works on any vendor whose profile says `image_inputs: True`. Groq's `gpt-oss-20b`
says `False`, which is why section 5's fallback can't take this ticket. Letting customers attach photos
is a budget decision before it's a feature: look at the input tokens.

## B. Explicit caching through LangChain (💳)

Implicit caching (section 6.3) needs nothing and caches part of the prefix. Explicit caching caches
all of it, for a time you choose, and bills you per hour for storing it. You create the cache with
Google's SDK and hand its name to LangChain as Gemini's own parameter. Flash-Lite needs a billed key
for it, so on a free key this cell explains itself instead of running.

In [42]:
try:
    cache = client.caches.create(model=settings.MODEL,
                                 config={"system_instruction": LIBRARY, "ttl": "300s"})
except Exception as err:
    cache = None
    print("explicit caching isn't available on this key:", type(err).__name__, str(err)[:120])

if cache:
    try:
        print("cached:", cache.usage_metadata.total_token_count, "tokens, for 5 minutes")
        cached_model = init_chat_model(settings.model_id, cached_content=cache.name)
        for question in ["How do I start a return?", "Who pays for return shipping?"]:
            usage = cached_model.invoke(f"Under ShopWise's policy: {question} One sentence.").usage_metadata
            print(f"input {usage['input_tokens']:6,}   cache_read {usage['input_token_details'].get('cache_read', 0):6,}")
    finally:
        client.caches.delete(name=cache.name)       # storage is billed until it expires or is deleted

cached: 24634 tokens, for 5 minutes


input 24,652   cache_read 24,634


input 24,651   cache_read 24,634


Every token of the library comes from the cache, from the first question on (implicit caching read
about 20,000 of the same prefix). The system prompt now lives *in* the cache, so the calls send only
the question: Gemini refuses a request that sets both (session 3, section 5.4). On Anthropic the
equivalent is `cache_control`, or `AnthropicPromptCachingMiddleware` once you've met middleware in
session 7.

## C. Async

The same interface has async versions (`ainvoke`, `abatch`, `astream`). Session 13's API server
uses them, so that one slow call doesn't block every other customer.

In [43]:
import asyncio


async def three_at_once():
    questions = [f"Name one colour in one word ({n})." for n in (1, 2, 3)]
    return await asyncio.gather(*(model.ainvoke(q) for q in questions))


replies = await three_at_once()        # Jupyter runs top-level await for you
print([r.text for r in replies])

['Blue', 'Blue', 'Red']


## D. One model object, many models

`init_chat_model` with no model creates one whose model you choose per call. Useful for comparing
models without rebuilding anything:

In [44]:
flexible = init_chat_model(configurable_fields=("model", "model_provider"))

for model_id in [settings.model_id, settings.FALLBACK]:
    provider, name = model_id.split(":", 1)
    r = flexible.invoke("Say hi in two words.",
                        config={"configurable": {"model": name, "model_provider": provider}})
    print(f"{model_id:38} {r.text.strip()}")

google_genai:gemini-3.5-flash-lite     Hello there!


groq:openai/gpt-oss-20b                Hi there.


## E. The schema's cost in full

Section 3.2's table measured one ticket. The ⭐⭐⭐ exercise does it properly: three or more vendors,
with and without the schema, then a smaller schema (shorter descriptions) measured again. Part of
that number is yours to shrink.

## F. Upgrading a pinned integration

`pyproject.toml` pins `langchain>=1.3,<2`: any 1.x, never 2.0 by surprise. Within that range
`uv lock` chose `langchain-google-genai` 4.3.2. Version 4.4.0 maps Gemini's errors onto the standard
exceptions of section 3.4 (`GoogleRateLimitError` is both a `ChatGoogleGenerativeAIError` and a
`ModelRateLimitError`). Before you bump a pin like that mid-project:

1. Read the release notes for every version you're skipping, not just the newest.
2. Look for behaviour changes, not only new features. "Now raises `X`" breaks an `except Y` written
   against the old class.
3. Bump one package, run your scoring script and your tests, and commit the lock file alone, so
   it's easy to revert.

## G. Why v1 exists

LangChain started in 2022 as a set of *chains*, fixed pipelines of prompts, models and parsers. Agents
were bolted on next (`AgentExecutor`), then every vendor and tool integration went into one huge
`langchain-community` package. By 2024 the lesson was clear: **components** (one interface to many
models) and **control flow** (what happens after what, with state, retries and human approval) are
different problems. LangGraph took control flow; `langchain-core` and the per-vendor packages took
components; and v1 (October 2025) cut `langchain` down to a small layer on top, built around
`init_chat_model` and `create_agent`. That's why so much code online looks unlike tonight's.